# 4. Construct the pinned builder

## Goal
Create the Docker toolchain used by both image construction and disposable VM tests.

## Prerequisites
Native amd64 Linux, Docker access, a reviewed CA, and a clean repository. **This operation uses privileged Docker later in the image build. Review the source before running it.**

In [ ]:
from pathlib import Path
import os, json, shutil, subprocess, shlex
START = Path.cwd().resolve()
ROOT = next((p for p in (START, *START.parents)
             if (p / "config/build.json").is_file()
             and (p / "notebooks/check.ipynb").is_file()), None)
assert ROOT is not None, "Open this notebook inside a clone of sc-linux"
print("SC Linux:", ROOT)

def command(*argv, enabled=False, env=None):
    args = [str(x) for x in argv]
    print("$", shlex.join(args))
    if not enabled:
        print("Preview only; opt in with the RUN_* variable above.")
        return None
    return subprocess.run(args, cwd=ROOT, env=env, check=True)

## Action 1: Inspect the toolchain definition

The builder is separate from the installed operating system. Docker, QEMU and Jupyter are not installed in the guest by this action.

In [ ]:
dockerfile = (ROOT / "Dockerfile").read_text()
print(dockerfile)
assert "mkosi" in dockerfile and "xorriso" in dockerfile and "qemu" in dockerfile

## Action 2: Identify the configured Debian snapshot

In [ ]:
config = json.loads((ROOT / "config/build.json").read_text())
print("Debian suite:", config["release"])
print("Snapshot:", config["snapshot"])
print("Expected mkosi package:", config["mkosi_package"])

## Action 3: Create the local Docker builder (opt-in)

`make builder` resolves a base image to a digest, builds a pinned Debian toolchain and records its local identity in `build/`.

In [ ]:
RUN_BUILDER = False
command("make", "builder", enabled=RUN_BUILDER)

## Action 4: Inspect recorded provenance

If the preceding build was not run, these files may not exist.

In [ ]:
for name in ("builder.id", "builder-base.txt"):
    path = ROOT / "build" / name
    print(f"{path.relative_to(ROOT)}: {path.read_text().strip() if path.is_file() else 'NOT YET CREATED'}")

## Check
The builder image ID is local to this Docker daemon. The base image digest identifies a reviewed input; by itself it does not make the complete build hermetic.

## Next
Continue to [5. Build the system image](05-image.ipynb).